# 📱 範例 04：選品電商—二手商品定價估算 (迴歸篇) — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本教材專為**「二手電商賣家、3C 轉賣估價師與選品小老闆（無程式背景）」**設計。
採用**「一段保證可執行的 Python 程式碼 ＋ 一段 AI 伴學互動提問指引」**的雙軌架構：
- 每一段程式碼都經過嚴謹測試，保證直接點擊即可順暢執行。
- 執行完程式碼後，**下方的文字儲存格**會以二手估價實務白話解釋程式在做什麼，並提供專屬的 **AI 提問 Prompt（提示詞）**，你可以複製貼給 Copilot、ChatGPT 或 Claude，讓 AI 替你深度解說商業決策邏輯！

---

## 📊 一、二手選品資料集介紹 (`used_items.csv`)

本數據集記錄了二手交易平台歷年 1,500 筆商品規格特徵與最終成交售價：

| 欄位名稱 | 資料型態 | 數值範例 / 類別 | 二手選品業務實務意涵 |
| :--- | :--- | :--- | :--- |
| **品類名稱** | 類別型 (`object`) | iPhone手機、電競筆電、數位單眼相機、機械腕錶、iPad平板 | 產品品項（流通流動性指標） |
| **原價_TWD** | 數值型 (`int64`) | 36,000 (元) | 當年全新品原廠官方定價 |
| **商品成色** | 類別型 (`object`) | 99成新、9成新、8成新、7成新 | 外觀成色磨損程度 |
| **使用年限_年** | 數值型 (`float64`) | 1.5 (年) | 出廠使用年數（核心折舊要素） |
| **配件完整度** | 類別型 (`object`) | 盒裝配件齊全、僅單機附副廠線、無任何配件 | 配件完整程度加成 |
| **品牌等級** | 類別型 (`object`) | 一線國際旗艦、主流知名大廠、新興平價品牌 | 品牌保值度指標 |
| **原廠保固剩餘月數** | 數值型 (`int64`) | 6 (個月) | 剩餘保固期 |
| **二手建議售價_TWD (目標 y)** | 數值型 (`int64`) | **22,500 (元)** | **預測目標：二手合理市場估值（連續數值迴歸）** |

---

## 🧭 二、機器學習全流程地圖

1. **環境配置與繁中字型**：確保圖表顯示中文，避免方塊亂碼 (`□□`)。
2. **資料載入與探索 (EDA)**：檢視品類原價分佈與折舊規律。
3. **特徵相關性與視覺化**：繪製使用年數與二手售價的散佈折舊趨勢圖。
4. **特徵工程與 Pipeline 建置**：8:2 切分訓練與測試集，嚴防資料洩漏。
5. **模型選型與 5 折交叉驗證**：比對「線性迴歸」與「隨機森林迴歸器」。
6. **測試集大考與收購毛利策略**：評估 R²、MAE，輸出模型並模擬新二手品估價。

---  
### 【步驟 0】：環境準備與繁體中文字型設定

In [ ]:
# [步驟 0 程式碼] 載入套件與設定繁體中文字型
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Heiti TC", "Microsoft JhengHei", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

print("✅ 套件與中文字型載入成功！準備進入二手商品估價分析。")

#### 💬【步驟 0 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  載入 Pandas 與繪圖庫，設定中文字型，確保後續繪製折舊圖表時標籤字體正常呈現。

---
💡 **AI 提問小抄：**

1. 📌 **折舊規律的機器學習建模**
   > 「二手商品通常第一年折舊最快（掉價 20~30%），之後折舊逐漸趨緩。這種非直線的折舊規律，在機器學習中應該如何透過特徵工程或非線性模型來捕捉？」


---  
### 【步驟 1】：載入資料與二手折舊結構探索 (EDA)

In [ ]:
# [步驟 1 程式碼] 讀取 used_items.csv 並檢視資料概況
df = pd.read_csv("used_items.csv")
print("資料集維度（列, 欄）：", df.shape)
print("\n--- 資料前 5 筆預覽 ---")
display(df.head())
print("\n--- 數值摘要統計 ---")
display(df.describe().round(1))

#### 💬【步驟 1 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  讀取資料集，檢查 1,500 筆二手物件的分佈，查看原價、使用年限與建議二手售價的數值範圍。

---
💡 **AI 提問小抄：**

1. 📌 **殘值率計算**
   > 「在二手選品中，我們常看『殘值率 = 二手售價 / 新品原價』。請幫我寫一行 Pandas 程式碼新增『殘值率』欄位，並統計各品類的平均殘值率？」


---  
### 【步驟 2】：成色等級與使用年數之折舊散佈圖繪製

In [ ]:
# [步驟 2 程式碼] 繪製成色等級與使用年數的二手價格分佈
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cond_order = ["99成新(僅拆封)", "9成新(輕微痕跡)", "8成新(正常使用)", "7成新(明顯磨損)"]
sns.boxplot(data=df, x="商品成色", y="二手建議售價_TWD", order=cond_order, palette="Oranges_r", ax=axes[0])
axes[0].set_title("不同成色等級之二手售價分佈")
axes[0].set_ylabel("二手售價 (TWD)")

sns.scatterplot(data=df, x="使用年限_年", y="二手建議售價_TWD", hue="品類名稱", alpha=0.7, ax=axes[1])
axes[1].set_title("使用年限 vs 二手售價 (依品類區分)")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

#### 💬【步驟 2 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  透過箱型圖比較 99 新與 7 成新的價格中位數落差，並以散佈圖觀察隨著使用年數增加，各類 3C 產品的價格向下折舊趨勢。

---
💡 **AI 提問小抄：**

1. 📌 **品類保值度比較**
   > 「從右側散佈圖可以看出，『iPhone手機』與『機械腕錶』在使用了 3 年後，售價依然高於其他品類。這在選品策略上代表什麼商業價值？」


---  
### 【步驟 3】：特徵工程與 Pipeline 建構

In [ ]:
# [步驟 3 程式碼] 切分訓練/測試集，並建立前處理管線
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df.drop(columns=["二手建議售價_TWD"])
y = df["二手建議售價_TWD"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

num_cols = ["原價_TWD", "使用年限_年", "原廠保固剩餘月數"]
cat_cols = ["品類名稱", "商品成色", "配件完整度", "品牌等級"]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
])

print(f"✅ 資料分割完成！訓練集：{X_train.shape[0]} 筆，測試集：{X_test.shape[0]} 筆")

#### 💬【步驟 3 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將原價、使用年限標準化，並將成色與配件類別文字轉換為虛擬變數，使用 Pipeline 完整保護訓練與測試集切分邊界。

---
💡 **AI 提問小抄：**

1. 📌 **序數編碼 vs 獨熱編碼**
   > 「『商品成色』本身有明顯順序大小（99新 > 9新 > 8新 > 7新），為什麼我們這裡用 `OneHotEncoder` 而不是 `OrdinalEncoder`？這兩種編碼方式在決策樹模型中有何優缺點？」


---  
### 【步驟 4】：模型選型與 5 折交叉驗證 (線性迴歸 vs 隨機森林)

In [ ]:
# [步驟 4 程式碼] 使用 5 折交叉驗證評估線性迴歸與隨機森林
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

pipe_lr = Pipeline([("preprocessor", preprocessor), ("regressor", LinearRegression())])
pipe_rf = Pipeline([("preprocessor", preprocessor), ("regressor", RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42))])

scores_lr = cross_val_score(pipe_lr, X_train, y_train, cv=5, scoring="r2")
scores_rf = cross_val_score(pipe_rf, X_train, y_train, cv=5, scoring="r2")

print(f"線性迴歸 5 折交叉驗證 平均 R²: {scores_lr.mean():.4f} (±{scores_lr.std():.4f})")
print(f"隨機森林 5 折交叉驗證 平均 R²: {scores_rf.mean():.4f} (±{scores_rf.std():.4f})")

#### 💬【步驟 4 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  運用 5 折交叉驗證比對線性迴歸與隨機森林，評估模型對於原價、年限與配件多重交互效應的擬合能力。

---
💡 **AI 提問小抄：**

1. 📌 **隨機森林在定價中的優勢**
   > 「二手定價往往充滿『門檻式』的跳躍（例如保固剩 1 個月跟完全過保，買家心理價差很大）。隨機森林的決策樹分裂機制是如何精確抓住這種斷崖式價格變化的？」


---  
### 【步驟 5】：測試集大考與指標解讀 (MAE, RMSE, R²)

In [ ]:
# [步驟 5 程式碼] 在全新測試集進行評估並繪製預測散佈圖
from sklearn.metrics import r2_score, mean_absolute_error

pipe_rf.fit(X_train, y_train)
y_pred = pipe_rf.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"=== 測試集最終評估結果 ===")
print(f"決定係數 R²: {r2:.4f} (模型解釋了 {r2*100:.1f}% 的二手價格變異)")
print(f"平均絕對誤差 MAE: ${mae:,.0f} 元 (平均估價誤差)")

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.5, color="#ea580c")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "k--", lw=2, label="完美預測對角線")
plt.title(f"二手商品真實成交價 vs 模型估算價 (MAE=${mae:,.0f})")
plt.xlabel("真實市場成交價 (TWD)")
plt.ylabel("模型預測估價 (TWD)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

#### 💬【步驟 5 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將模型應用於 300 筆全新測試商品，輸出 MAE 與實際值 vs 預測值對角線圖，直觀檢視模型在各價格帶的估價精準度。

---
💡 **AI 提問小抄：**

1. 📌 **估價誤差對毛利的影響**
   > 「如果我們的 MAE 平均誤差是 1,200 元，二手店家在收購時應該設定幾折的安全邊際，才能確保即使模型稍微高估了市場價，店家也不會賠本收購？」


---  
### 【步驟 6】：新二手物件即時估價 (Inference) 與收購毛利策略

In [ ]:
# [步驟 6 程式碼] 保存模型並示範新二手商品即時估價
joblib.dump(pipe_rf, "model.joblib")
print("✅ 模型管線已成功保存至 model.joblib！")

# 模擬客人帶了一台二手 iPhone 來店裡估價
incoming_item = pd.DataFrame([{
    "品類名稱": "iPhone手機",
    "原價_TWD": 38900,
    "商品成色": "9成新(輕微痕跡)",
    "使用年限_年": 1.0,
    "配件完整度": "盒裝配件齊全",
    "品牌等級": "一線國際旗艦",
    "原廠保固剩餘月數": 2
}])

loaded_model = joblib.load("model.joblib")
suggested_price = loaded_model.predict(incoming_item)[0]
suggested_buyin = suggested_price * 0.75

print(f"\n🔍 二手進貨即時估價報告：")
print(f"終端市場預估建議售價：${suggested_price:,.0f} 元")
print(f"店家建議收購報價上限（保留25%翻新毛利）：${suggested_buyin:,.0f} 元")

#### 💬【步驟 6 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將模型管線存檔，並模擬店家櫃台即時輸入客人手機規格，自動算出建議零售價與店家安全收購價。

---
💡 **AI 提問小抄：**

1. 📌 **自動化爬蟲行情連動**
   > 「如果我想讓這個模型每天自動結合蝦皮或旋轉拍賣的即時掛牌價，在推論前自動微調價格權重，這在現代 MLOps 架構中該如何設計？」
